# 01 · Contratos de datos, granularidad y auditoría

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## Llaves, multiplicidad y cobertura

Sea $K=(loan\_key,merchant\_key,fecha\_corte)$. La unicidad requiere
$$\forall k,\quad n(k)=1.$$ 
La tasa de duplicados es $r_{dup}=1-\lvert\mathrm{unique}(K)\rvert/N$; no equivale al porcentaje de filas repetidas. La cobertura de una variable $x$ es $C_x=\sum_i \mathbf1(x_i\neq\mathrm{NA})/N$. **Nunca convertir ausencia de observación en DPD=0**.

In [ ]:
df=load('plantilla_panel_mensual.csv')
keys=['loan_key','merchant_key','fecha_corte']
assert set(keys).issubset(df.columns)
if not df.empty:
    assert not df[keys].isna().any().any(), 'Claves nulas'
    print('Duplicados por clave:',df.duplicated(keys).sum())
    display(df.notna().mean().sort_values().to_frame('cobertura').head(20))
else: print('PENDIENTE: extracto real del panel mensual')

## Consistencia de saldos, DPD y unidades

Definir $B\ge0$ para saldo de capital ordinario, con excepciones documentadas por ajustes contables. Si $L>0$, utilización $U=B/L$; valores $U>1$ pueden ser legítimos (intereses, sobregiros) o defectos de calidad y requieren revisión. Para dos fuentes, error de conciliación:
$$\epsilon_t=\sum_i B^{panel}_{i,t}-\sum_i B^{contabilidad}_{i,t},\quad \delta_t=\epsilon_t/\max(1,\lvert\sum_i B^{contabilidad}_{i,t}\rvert).$$

In [ ]:
if not df.empty:
    for c in ['mora','saldo_capital','cupo_vigente']:
        if c in df: df[c]=pd.to_numeric(df[c],errors='coerce')
    print('DPD negativos:',int((df['mora']<0).sum()) if 'mora' in df else 'sin columna')
    if {'saldo_capital','cupo_vigente'}.issubset(df):
        print('Cupos no positivos:',int((df.cupo_vigente<=0).sum()))
        display(df[['saldo_capital','cupo_vigente']].describe().T)
else: print('PENDIENTE: reconciliar con balance contable')

## Integridad temporal y prevención de fuga

Para un predictor observado en fecha de decisión $t_0$, exigir $t_{disponibilidad}(x)\le t_0$. Para un snapshot $t$, exigir $t_{disponibilidad}(x)\le t$. Los atributos con vigencia $[a,b)$ se unen mediante $a\le t<b$. Una fecha de evento anterior al corte **no prueba** que el dato estuviera publicado entonces.

In [ ]:
cols=list(df.columns)
for expected in ['fecha_corte','fecha_desembolso','fecha_disponibilidad','loan_key','merchant_key']:
    print(f'{expected}:', 'presente' if expected in cols else 'no presente; evaluar incorporación')

## Pruebas de calidad por corte

La variación de población es $g_t=N_t/N_{t-1}-1$. Un salto no implica error: contrastar contra altas, bajas, castigos, migraciones y cambios de cobertura. Evaluar PSI por variable numérica, con bins de referencia fijos:
$$PSI=\sum_{b=1}^{B}(p_b-q_b)\log(p_b/q_b).$$
Aplicar suavizado $\varepsilon$ cuando alguna proporción sea cero; los umbrales de alerta deben calibrarse, no asumirse universales.

In [ ]:
if not df.empty:
    v=df.groupby('fecha_corte').agg(n=('loan_key','nunique')).reset_index()
    v['cambio_pct']=v.n.pct_change()*100
    display(v.tail(15))
    ax=v.plot(x='fecha_corte',y='n',title='Cobertura de créditos por corte',legend=False)
    ax.set_ylabel('Créditos únicos');plt.tight_layout();plt.show()
else: print('PENDIENTE: serie de cobertura')

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.